<a href="https://colab.research.google.com/github/agil-p-varghese/Fine-Tunning--Models/blob/main/Fine_tunning_model_V2(with_eval).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Clean install of Axolotl and essential visualization libraries
!pip install --no-build-isolation "axolotl[gpl]>=0.20.0"
!pip install matplotlib seaborn
print("Environment dependencies installed successfully!")

In [ ]:
import json

# 150 Training samples
train_data = [
    {"instruction": "Respond according to the support response policy.", "input": "I've tried everything and your application still doesn't work.", "output": "I'm sorry you're still experiencing the issue. Please share the error message you're seeing so we can determine the next troubleshooting step."},
    {"instruction": "Respond according to the support response policy.", "input": "Can you tell me when my refund will arrive?", "output": "I can help check the refund status. Please provide your order or transaction number."},
    {"instruction": "Respond according to the support response policy.", "input": "Thanks, it's working now!", "output": "You're welcome! I'm glad the issue is resolved."},
    {"instruction": "Respond according to the support response policy.", "input": "I have ordered an iPhone but I was delivered a clock. What should I do?", "output": "Sorry for the inconvenience. Please share your order ID and a screenshot of your payment so we can rectify it."},
    {"instruction": "Respond according to the support response policy.", "input": "I want to place a bulk order for my shop and speak with your manager.", "output": "We can arrange a telephone call with you for discussion. Please provide your telephone number and available time."},
    {"instruction": "Respond according to the support response policy.", "input": "I received a low quality product and want to raise a complaint.", "output": "I'm sorry for your experience. Please contact the product manager at productqueries@myshpApp for raising a complaint."},
    {"instruction": "Respond according to the support response policy.", "input": "I want to change my delivery address in the app.", "output": "Yes, I can guide you. Please go to the personal details tab on the bottom right of the home page of the application."},
    {"instruction": "Respond according to the support response policy.", "input": "I want to cancel a discounted item order and get a refund.", "output": "I'm very sorry to inform you that we do not provide any refund for items bought at a discount, but you can cancel the order."},
    {"instruction": "Respond according to the support response policy.", "input": "It has been 20 days since delivery. Can I return the order and apply for a refund?", "output": "Sorry, we don't accept return requests after 9 days of delivery."},
    {"instruction": "Respond according to the support response policy.", "input": "How much time will it take to process the refund?", "output": "Under normal conditions, it will take 10 working days. If you provide your refund request ID, I could check the status."},
    {"instruction": "Respond according to the support response policy.", "input": "When does your app give great discounts?", "output": "We provide goods at discounted prices on Sree Narayana Guru Jayanti day (October 29)."}
] * 14  # scales to ~150 rows

# 30 Validation samples
val_data = [
    {"instruction": "Respond according to the support response policy.", "input": "Can I return a discounted item?", "output": "I'm sorry, but items bought at a discount sale are not eligible for returns."},
    {"instruction": "Respond according to the support response policy.", "input": "My product worth 6000 stopped working after 40 days. Is it covered?", "output": "Yes, products above 5000 come with a two-month warranty. Please share your order ID so we can assist you with the repair or replacement."}
] * 15  # scales to ~30 rows

# 40 Test samples (held out)
test_data = [
    {"instruction": "Respond according to the support response policy.", "input": "I want to return an item delivered 12 days ago.", "output": "Sorry, we only accept return requests within 9 days of product delivery."},
    {"instruction": "Respond according to the support response policy.", "input": "My item arrived completely damaged! What compensation do I get?", "output": "I am so sorry your product arrived damaged. You will be refunded the complete bill amount along with a complimentary 200 rupees coupon. Please share your order ID."},
    {"instruction": "Respond according to the support response policy.", "input": "Where should I mail regarding a product query?", "output": "You can mail your product-related queries to productqueries@myshpApp."},
    {"instruction": "Respond according to the support response policy.", "input": "Who should I contact for app bugs?", "output": "For app-related queries, you can contact the application manager."}
] * 10  # scales to ~40 rows

# Write out datasets to JSONL files
for filename, data in [("train.jsonl", train_data), ("val.jsonl", val_data), ("test.jsonl", test_data)]:
    with open(filename, "w") as f:
        for item in data:
            f.write(json.dumps(item) + "\n")

print(f"Datasets created successfully: Train ({len(train_data)}), Val ({len(val_data)}), Test ({len(test_data)})")

In [ ]:
yaml_content = """
base_model: Qwen/Qwen2.5-3B-Instruct
model_type: AutoModelForCausalLM
tokenizer_type: AutoTokenizer

load_in_4bit: true
adapter: qlora
lora_r: 16
lora_alpha: 32
lora_dropout: 0.05
lora_target_modules:
  - q_proj
  - v_proj
  - k_proj
  - o_proj

datasets:
  - path: /content/train.jsonl
    type: alpaca

datasets_val:
  - path: /content/val.jsonl
    type: alpaca

dataset_prepared_path: /content/prepared_data
val_set_size: 0.15
output_dir: ./support_bot_out_v2

sequence_len: 512
sample_packing: false
micro_batch_size: 2
gradient_accumulation_steps: 4
epochs: 2
learning_rate: 2e-4
optimizer: adamw_torch
lr_scheduler: cosine

save_steps: 10
logging_steps: 2
bf16: true
"""

with open("support_bot.yml", "w") as f:
    f.write(yaml_content.strip())

print("support_bot.yml configured successfully!")

In [ ]:
!rm -rf prepared_data support_bot_out_v2

# Direct python execution bypasses CLI wrapper conflicts
!python -m axolotl.cli.train support_bot.yml

In [ ]:
import os
import json
import matplotlib.pyplot as plt

log_path = "./support_bot_out_v2/trainer_state.json"

if os.path.exists(log_path):
    with open(log_path, "r") as f:
        trainer_state = json.load(f)

    steps, train_losses, eval_steps, eval_losses = [], [], [], []
    for entry in trainer_state.get("log_history", []):
        if "loss" in entry:
            steps.append(entry["step"])
            train_losses.append(entry["loss"])
        if "eval_loss" in entry:
            eval_steps.append(entry["step"])
            eval_losses.append(entry["eval_loss"])

    plt.figure(figsize=(8, 5))
    if steps:
        plt.plot(steps, train_losses, label="Training Loss", marker="o")
    if eval_losses:
        plt.plot(eval_steps, eval_losses, label="Validation Loss", marker="x", color="red")
    plt.xlabel("Steps")
    plt.ylabel("Loss")
    plt.title("Training and Validation Loss Curve")
    plt.legend()
    plt.grid(True)
    plt.show()
else:
    print("Trainer state log not found. Ensure training finished successfully.")

In [ ]:
import torch
import gc
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

# Clear memory
for var_name in ['model', 'base_model', 'tokenizer']:
    if var_name in locals(): del locals()[var_name]
torch.cuda.empty_cache()
gc.collect()

base_model_id = "Qwen/Qwen2.5-3B-Instruct"
adapter_path = "./support_bot_out_v2"

print("Loading tokenizer and models...")
tokenizer = AutoTokenizer.from_pretrained(base_model_id)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id, dtype=torch.bfloat16, device_map="cuda:0"
)
ft_model = PeftModel.from_pretrained(base_model, model_id=adapter_path)
ft_model.eval()

def generate_text(model_inst, prompt):
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to("cuda") for k, v in inputs.items()}
    with torch.no_grad():
        out = model_inst.generate(
            **inputs, max_new_tokens=80, temperature=0.1, do_sample=True, pad_token_id=tokenizer.pad_token_id
        )
    return tokenizer.decode(out[0], skip_special_tokens=True).split("### Response:")[-1].strip()

print("\n" + "="*80)
print("EVALUATION & COMPARISON: BASE MODEL VS. FINE-TUNED BOT")
print("="*80)

for idx, sample in enumerate(test_data[:3], 1):
    customer_query = sample["input"]
    prompt = (
        "Below is an instruction that describes a task, paired with an input that provides further context. "
        "Write a response that appropriately completes the request.\n\n"
        "### Instruction:\nRespond according to the support response policy.\n\n"
        f"### Input:\n{customer_query}\n\n"
        "### Response:\n"
    )

    # Base Model generation (disabling adapter weights temporarily)
    with ft_model.disable_adapter():
        base_resp = generate_text(ft_model, prompt)

    # Fine-Tuned Model generation
    ft_resp = generate_text(ft_model, prompt)

    print(f"\n[Test Case {idx}] Customer: {customer_query}")
    print(f"  * Base Model:       {base_resp}")
    print(f"  * Fine-Tuned Bot:   {ft_resp}")
    print(f"  * Expected Policy:  {sample['output']}")
    print("-" * 80)